# Multimodal AI — Blood Work Analysis

In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langchain.tools import tool
from langchain.agents import create_agent
import base64

load_dotenv()

C:\Code\agentic-ai-crash-course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Code\agentic-ai-crash-course\.venv\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


True

## Encode the image and send to the vision model

In [2]:
with open("blood_work.png", "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode()

llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct")

message = HumanMessage(content=[
    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
    {"type": "text",      "text": "This is a blood work report. Extract all test results and flag any values outside the normal range."}
])

response = llm.invoke([message])
print(response.content)

**Patient:** Rajesh Sharma, Age 48, Male
**Date:** May 7, 2026

### **COMPLETE BLOOD COUNT (CBC)**

*   **Hemoglobin:** 15.1 g/dL **(Normal)**
    *   Normal Range: 13.5-17.5 g/dL
*   **Hematocrit:** 44% **(Normal)**
    *   Normal Range: 41-53%
*   **WBC:** 6.8 x10^3/uL **(Normal)**
    *   Normal Range: 4.5-11.0 x10^3/uL
*   **Platelets:** 220 x10^3/uL **(Normal)**
    *   Normal Range: 150-400 x10^3/uL

### **LIPID PANEL**

*   **Total Cholesterol:** 238 mg/dL **(Elevated)**
    *   Normal: <200 mg/dL
*   **LDL Cholesterol:** 162 mg/dL **(Elevated)**
    *   Normal: <100 mg/dL
*   **HDL Cholesterol:** 36 mg/dL **(Low)**
    *   Normal: >40 mg/dL
*   **Triglycerides:** 188 mg/dL **(Elevated)**
    *   Normal: <150 mg/dL

### **METABOLIC PANEL**

*   **Glucose (Fasting):** 92 mg/dL **(Normal)**
    *   Normal Range: 70-99 mg/dL
*   **HbA1c:** 5.3% **(Normal)**
    *   Normal: <5.7%
*   **Creatinine:** 1.0 mg/dL **(Normal)**
    *   Normal Range: 0.7-1.3 mg/dL
*   **eGFR:** 82 mL/min *

## Suggest Diet Plan Agent

The agent reads the blood work image, categorises the condition, then calls the diet tool.

In [3]:
@tool
def get_diet_recommendation(condition: str) -> dict:
    """Given a health condition, returns a diet plan. Condition must be one of: normal, high_cholesterol, high_sugar."""
    diet_plans = {
        "high_cholesterol": {
            "eat":        ["fruits", "vegetables", "whole grains", "lean protein"],
            "do_not_eat": ["red meat", "fried food", "full-fat dairy", "processed snacks"],
        },
        "high_sugar": {
            "eat":        ["vegetables", "whole grains", "legumes", "nuts"],
            "do_not_eat": ["white rice", "white sugar", "junk food", "sugary drinks"],
        },
        "normal": {
            "eat":        ["vegetables", "fruits", "whole grains", "lean protein"],
            "do_not_eat": ["excessive sugar", "processed food", "trans fats"],
        },
    }
    return diet_plans.get(condition, diet_plans["normal"])

In [4]:
SYSTEM_PROMPT = """
You are a helpful medical and nutrition assistant.
For the input blood work image, extract the numbers and the normal range, then categorize
the condition as one of: normal, high_cholesterol, high_sugar.
Then call the appropriate tool to retrieve and present the diet plan.
"""

diet_agent = create_agent(
    llm,
    tools=[get_diet_recommendation],
    system_prompt=SYSTEM_PROMPT,
)

In [5]:
result = diet_agent.invoke({
    "messages": [HumanMessage(content=[
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
        {"type": "text",      "text": "Analyse this blood work report and suggest a diet plan."},
    ])]
})

print(result["messages"][-1].content)

The patient, Rajesh Sharma, has high cholesterol. A diet plan for someone with high cholesterol includes eating foods low in saturated and trans fats, such as fruits, vegetables, whole grains, and lean protein. Foods to avoid include red meat, fried foods, full-fat dairy products, and processed snacks.
